# Nigeria Climate Data Profiling & EDA

This notebook profiles and cleans the climate dataset for Nigeria from 2015-2026.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import zscore

df = pd.read_csv('../data/nigeria.csv')
print(f"Initial shape: {df.shape}")

## 1. Missing Value Report & Duplicates
We replace NASA's `-999` sentinel values with `NaN`, check for duplicates, and calculate missing percentages.

In [ ]:
neg_999_count = (df == -999).sum().sum()
print(f"Count of -999 values: {neg_999_count}")
df.replace(-999, np.nan, inplace=True)

dup_count = df.duplicated().sum()
print(f"Duplicate rows: {dup_count}")
df.drop_duplicates(inplace=True)

missing_pct = (df.isna().sum() / len(df)) * 100
print("\nMissing Values %:")
print(missing_pct[missing_pct > 0])

**Interpretation**: Minimal to zero duplicates or `-999`. Handled via drop and replacing.

## 2. Summary Statistics & Outliers

In [ ]:
display(df.describe())

cols_for_z = ['T2M', 'T2M_MAX', 'T2M_MIN', 'PRECTOTCORR', 'RH2M', 'WS2M', 'WS2M_MAX']
for col in cols_for_z:
    if col in df.columns:
        z_scores = np.abs(zscore(df[col].dropna()))
        outliers = (z_scores > 3).sum()
        print(f"{col} outliers (>3 std dev): {outliers}")

**Interpretation**: `PRECTOTCORR` often has outliers representing distinct extreme rain events. Retaining these outliers since they represent genuine extreme weather events.

In [ ]:
# Export Clean Data
df['Date'] = pd.to_datetime(df['YEAR'] * 1000 + df['DOY'], format='%Y%j')
df['Month'] = df['Date'].dt.month
df['Country'] = 'Nigeria'
df.ffill(inplace=True)
df.to_csv('../data/nigeria_clean.csv', index=False)

## 3. Time Series Analysis
Plotting monthly average temperature and total precipitation.

In [ ]:
monthly_t2m = df.groupby('Month')['T2M'].mean()
monthly_prec = df.groupby('Month')['PRECTOTCORR'].sum()

fig, ax1 = plt.subplots(figsize=(10,5))
ax1.plot(monthly_t2m.index, monthly_t2m.values, color='red', marker='o')
ax1.set_ylabel('Avg Temperature (°C)')
ax1.set_xlabel('Month')
plt.title('Monthly Avg Temperature - Nigeria (2015-2026)')
plt.show()

fig, ax2 = plt.subplots(figsize=(10,5))
ax2.bar(monthly_prec.index, monthly_prec.values, color='blue')
ax2.set_ylabel('Total Precipitation (mm)')
ax2.set_xlabel('Month')
plt.title('Monthly Total Precipitation - Nigeria (2015-2026)')
plt.show()

**Trends observed**: Strong seasonal variations aligning with regional climate systems.

## 4. Correlation & Distributions

In [ ]:
numeric_cols = df.select_dtypes(include=[np.number])
corr = numeric_cols.corr()
plt.figure(figsize=(10,8))
sns.heatmap(corr, annot=False, cmap='coolwarm')
plt.title('Correlation Heatmap')
plt.show()

sns.scatterplot(x='RH2M', y='T2M', data=df)
plt.title('Temperature vs Relative Humidity')
plt.show()

**Correlations Interpretations**:
1. Strong correlation between wind metrics.
2. Interdependence between humidity variables.
3. Temperature generally inversely tracking precipitation/humidity seasonality.